# 02 · Spatiotemporal observation exploration

Check pass independence and early clusters. Most apparent satellite "motion" can reflect sampling cadence or clouds, so clusters receive reduced correlated evidence weights.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import json
import matplotlib.pyplot as plt
INPUT = ROOT / "data/synthetic/toy_firms.csv"  # SYNTHETIC DEMO, NOT historical NASA data
WEATHER = ROOT / "data/synthetic/toy_weather.json"
assert INPUT.exists(), "Run `python scripts/make_synthetic.py` from repository root first"
print("Data mode: SYNTHETIC - demonstration only")

In [ ]:
from wildfire.ingest import load_firms, load_weather
from wildfire.features import cluster_detections,weather_summary
detections = load_firms(INPUT)
weather = load_weather(WEATHER)
clusters = cluster_detections(detections,max_hours=10)
print('Independent cluster count:', len(clusters))
for c in clusters: print(c['cluster_id'], c['timestamp_utc'], c['n_pixels'],'pixels')

In [ ]:
# Inspect rather than assume every hot spot belongs to the target incident.
fig, ax = plt.subplots(figsize=(9,5))
s = ax.scatter([d['longitude'] for d in detections], [d['latitude'] for d in detections],
               c=list(range(len(detections))), cmap='inferno', s=28)
ax.scatter([c['longitude'] for c in clusters], [c['latitude'] for c in clusters],
           marker='x',s=110,c='cyan',label='Overpass clusters')
ax.set(xlabel='Longitude',ylabel='Latitude',title='Synthetic data: raw detections and cluster centres')
ax.legend();plt.colorbar(s,ax=ax,label='Acquisition order');plt.show()

In [ ]:
from wildfire.ingest import parse_utc
from datetime import timedelta
first = parse_utc(clusters[0]['timestamp_utc'])
last = first + timedelta(hours=10)
summary = weather_summary(weather,first.isoformat().replace('+00:00','Z'),
                           last.isoformat().replace('+00:00','Z'))
print(json.dumps(summary,indent=2))

**Decision gate:** If all thermal clusters belong to one late overpass, causal backtracking is underdetermined. Increase temporal cadence using GOES or change fire rather than manufacture certainty.